In [1]:
# 导入所需库
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score # [新增] 用于计算宏平均召回率
import datetime
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 
os.environ['PROJ_LIB'] = r'D:\ProgramData\Anaconda3\envs\tensorflow210\Library\share\proj'
import csv

D:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\tensorflow\python\framework\dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
D:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\tensorflow\python\framework\dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
D:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\tensorflow\python\framework\dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
D:\ProgramData\Anaconda3\envs\tensorflow210\lib\s

## 读取与预处理数据

In [2]:
import pandas as pd
import numpy as np
import rasterio
import glob
import os
from rasterio.warp import transform

# ================= 配置区 =================
# 1. 简化的训练点位表 (必须包含 Index, Alliance, X, Y)
# X, Y 必须是 WGS84 (经纬度) 坐标
simple_train_csv = r'F:/TensorFlow/xinjiang/traindata20250626_3.csv'

# 2. 遥感影像文件夹 (包含那 ~250 个单波段 tif)
tif_folder = r'F:\TensorFlow\Xinjiang_2'

# 3. 类别映射表 (保持不变)
mapping_csv = r'F:/TensorFlow/xinjiang/traindata20250626_2_class_mapping.csv'
# ==========================================

print("正在读取训练点位数据...")
points_df = pd.read_csv(simple_train_csv)

# 确保列名正确，如果不包含 Index 可以自动生成
if 'Index' not in points_df.columns:
    points_df['Index'] = range(len(points_df))

# 获取 TIF 文件列表 (必须排序，这是特征对齐的灵魂)
tif_files = sorted(glob.glob(os.path.join(tif_folder, '*.tif')))
print(f"检测到 {len(tif_files)} 个特征影像文件。")

# ==== 步骤 1: 坐标转换 (WGS84 -> 影像投影) ====
# 读取第一幅影像作为参考坐标系
with rasterio.open(tif_files[0]) as src:
    dst_crs = src.crs  # 影像的坐标系
    
print("正在进行坐标转换 (WGS84 -> 影像投影坐标)...")
# 假设 X是经度, Y是纬度。WGS84 的 EPSG 代码是 4326
# rasterio.warp.transform 输入是 (src_crs, dst_crs, xs, ys)
# 注意：transform 返回的是 tuple (new_xs, new_ys)
tgt_x, tgt_y = transform({'init': 'epsg:4326'}, dst_crs, points_df['X'].values, points_df['Y'].values)

# 将转换后的坐标生成 sample 列表: [(x1, y1), (x2, y2), ...]
sample_coords = list(zip(tgt_x, tgt_y))

# ==== 步骤 2: 批量提取像素值 ====
print("开始从影像中提取特征值 (这可能需要几分钟)...")

feature_data = {} # 用于暂存提取结果

# 遍历所有 TIF 文件
for idx, tif_path in enumerate(tif_files):
    file_name = os.path.basename(tif_path)
    # 用文件名作为特征列名，方便排查
    col_name = file_name 
    
    if (idx + 1) % 10 == 0:
        print(f"  正在处理第 {idx + 1}/{len(tif_files)} 个影像...")

    with rasterio.open(tif_path) as src:
        # src.sample() 是一个生成器，我们需要将其转为 list
        # 结果形式是 [[val], [val], ...] 所以要取 [0]
        # 这里的 sample_coords 已经是转换好的坐标了
        values = [val[0] for val in src.sample(sample_coords)]
        feature_data[col_name] = values

# 将特征字典转换为 DataFrame
features_df = pd.DataFrame(feature_data)

# ==== 步骤 3: 组装最终的 train_data ====
print("正在组装训练数据...")

# 合并 基础信息(Index, Alliance) 和 提取的特征(features_df)
# 只要行索引一致，直接 concat 即可
train_data = pd.concat([points_df[['Index', 'Alliance']], features_df], axis=1)

# 处理无效值（如果点位超出了影像范围，提取值可能是 nodata）
# 假设 nodata 是 -9999 或 NaN，这里做简单清洗
# 根据你的影像实际情况修改，如果全是有效点可跳过
train_data = train_data.replace(-9999, np.nan).dropna()

# ==== 步骤 4: 原始预处理流程衔接 (保持你原有的逻辑) ====
class_mapping = pd.read_csv(mapping_csv)

# 合并class_mapping
train_data = train_data.merge(class_mapping[['Alliance', 'num', 'Formation']], on='Alliance', how='left')

# 自动识别特征列：除了 Index, Alliance, num, Formation, X, Y 之外的都是特征
feature_cols = [col for col in train_data.columns if col not in ['Index', 'Alliance', 'num', 'Formation']]

print(f"最终训练数据构建完成: {train_data.shape}")
print(f"特征列数量: {len(feature_cols)} (应与影像数一致)")

X = train_data[feature_cols]
num_labels = train_data['num']
formation_labels = train_data['Formation']

# 编码大类标签
formation_encoder = LabelEncoder()
formation_y = formation_encoder.fit_transform(formation_labels)

X_train = X
num_train = num_labels
formation_train = formation_labels
formation_y_train = formation_y

# 采用分层抽样(stratify=num_labels)，保证训练集和验证集的各小类占比基本一致
X_train, X_val, num_train, num_val, formation_y_train, formation_y_val = train_test_split(
    X, num_labels, formation_y, test_size=0.2, random_state=42, stratify=num_labels
)

print(f"数据准备完毕。训练集样本: {len(X_train)}, 验证集样本: {len(X_val)}")

正在读取训练点位数据...
检测到 187 个特征影像文件。
正在进行坐标转换 (WGS84 -> 影像投影坐标)...
开始从影像中提取特征值 (这可能需要几分钟)...
  正在处理第 10/187 个影像...
  正在处理第 20/187 个影像...
  正在处理第 30/187 个影像...
  正在处理第 40/187 个影像...
  正在处理第 50/187 个影像...
  正在处理第 60/187 个影像...
  正在处理第 70/187 个影像...
  正在处理第 80/187 个影像...
  正在处理第 90/187 个影像...
  正在处理第 100/187 个影像...
  正在处理第 110/187 个影像...
  正在处理第 120/187 个影像...
  正在处理第 130/187 个影像...
  正在处理第 140/187 个影像...
  正在处理第 150/187 个影像...
  正在处理第 160/187 个影像...
  正在处理第 170/187 个影像...
  正在处理第 180/187 个影像...
正在组装训练数据...
最终训练数据构建完成: (1391, 191)
特征列数量: 187 (应与影像数一致)
数据准备完毕。训练集样本: 1112, 验证集样本: 279


## 构建神经网络与损失函数

In [3]:
def build_model(input_dim, num_classes, use_bn_dropout=True):
    layers = [tf.keras.layers.Dense(256, activation='relu', input_dim=input_dim, kernel_regularizer=regularizers.l2(0.001))]
    if use_bn_dropout:
        layers.extend([tf.keras.layers.BatchNormalization(), tf.keras.layers.Dropout(0.3)])
        
    layers.append(tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
    if use_bn_dropout:
        layers.extend([tf.keras.layers.BatchNormalization(), tf.keras.layers.Dropout(0.3)])
        
    layers.append(tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
    if use_bn_dropout:
        layers.extend([tf.keras.layers.BatchNormalization(), tf.keras.layers.Dropout(0.3)])
        
    layers.append(tf.keras.layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.001)))
    if use_bn_dropout:
        layers.extend([tf.keras.layers.BatchNormalization(), tf.keras.layers.Dropout(0.3)])
        
    layers.append(tf.keras.layers.Dense(num_classes, activation='softmax'))
    
    model = tf.keras.Sequential(layers)
    return model

def multi_category_focal_loss2(gamma=2., alpha=.25, class_weights=None):
    epsilon = 1.e-7
    gamma = float(gamma)
    alpha = tf.constant(alpha, dtype=tf.float32)
    if class_weights is not None:
        weights = np.array([class_weights.get(i, 1.0) for i in range(len(class_weights))])
        class_weights_tf = tf.constant(weights, dtype=tf.float32)
    else:
        class_weights_tf = None
    def focal_loss_fixed(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        y_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        ce = -tf.math.log(y_t)
        weight = tf.pow(1. - y_t, gamma)
        fl = alpha_t * weight * ce
        if class_weights_tf is not None:
            fl = fl * class_weights_tf
        return tf.reduce_mean(fl)
    return focal_loss_fixed

# [修改] 增加 quiet 参数，控制是否打印内部过程
def train_and_evaluate(x_train, y_train, num_classes, x_val=None, y_val=None, loss_type='focal', use_bn_dropout=True, verbose=0, quiet=False):
    unique_classes, class_counts = np.unique(y_train, return_counts=True)
    majority_class_count = np.max(class_counts)
    baseline_acc = majority_class_count / len(y_train)
    
    if not quiet:
        print(f"  [基准提示] 训练集盲猜基准: {baseline_acc:.4f}")

    y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
    
    val_data = None
    if x_val is not None and y_val is not None:
        y_val_cat = tf.keras.utils.to_categorical(y_val, num_classes=num_classes)
        val_data = (x_val, y_val_cat)

    class_weights_array = compute_class_weight(class_weight='balanced', classes=np.arange(num_classes), y=y_train)
    class_weights = dict(enumerate(class_weights_array))
    
    model = build_model(x_train.shape[1], num_classes, use_bn_dropout)
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
    
    if loss_type == 'focal':
        loss_fn = multi_category_focal_loss2(alpha=0.25, gamma=2, class_weights=class_weights)
    else:
        loss_fn = 'categorical_crossentropy'

    model.compile(loss=loss_fn, optimizer=optimizer, metrics=['accuracy'])
    early_stopping = EarlyStopping(monitor='loss', patience=30, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=10, verbose=0)
    
    history = model.fit(x_train, y_train_cat, validation_data=val_data, epochs=500, batch_size=64, verbose=verbose, callbacks=[reduce_lr, early_stopping])
    
    n_epochs = len(history.history['loss'])
    final_loss = history.history['loss'][-1]
    final_acc = history.history['acc'][-1]
    improvement = final_acc - baseline_acc
    
    if not quiet:
        print(f"训练轮数: {n_epochs}, 最终训练精度: {final_acc:.4f} (比基准提升: {improvement:+.4f}), 损失: {final_loss:.4f}")
    
    if val_data is not None:
        val_loss, val_acc = model.evaluate(x_val, y_val_cat, verbose=0)
        model._val_acc = val_acc  # [重点] 绑定到模型对象上，方便外部提取
        if not quiet:
            print(f"  >>> 独立验证集精度: {val_acc:.4f} <<<")
    else:
        model._val_acc = None
        
    return model

## 训练分类模型

In [4]:
# 获取当前日期字符串
save_date = datetime.datetime.now().strftime('%Y%m%d')
model_dir = Path(f'F:/TensorFlow/xinjiang/models{save_date}')
model_dir.mkdir(parents=True, exist_ok=True)

# ==== 工具函数 ====
def get_eng_formation_map(class_mapping_path, class_mapping_df):
    if 'Eng_Formation' not in class_mapping_df.columns:
        class_mapping_df = pd.read_csv(class_mapping_path)
    return dict(zip(class_mapping_df['Formation'], class_mapping_df['Eng_Formation']))

def save_model(model, save_dir, name):
    name = name.replace(' ', '')
    path = Path(save_dir) / f'{name}.h5'
    model.save(str(path))
    print(f"模型已保存到: {path}")
    return path

# [修改] 全面引入验证集参数，自动记录 single_class_map 用于预测时调用
def train_and_save_all_models(X_train, num_train, formation_y_train, 
                              X_val, num_val, formation_y_val, 
                              formation_encoder, eng_formation_map, save_dir):
    
    print("="*60)
    print(f"[全局基准提示] 训练集全局盲猜小类基准: {np.max(np.unique(num_train, return_counts=True)[1]) / len(num_train):.4f}")
    if len(num_val) > 0:
        print(f"[全局基准提示] 验证集全局盲猜小类基准: {np.max(np.unique(num_val, return_counts=True)[1]) / len(num_val):.4f}")
    print("="*60)

    # 1. 训练大类模型
    num_formation_classes = len(np.unique(formation_y_train))
    print(f"大类（Formation）类别数: {num_formation_classes}")
    formation_model = train_and_evaluate(X_train, formation_y_train, num_formation_classes, x_val=X_val, y_val=formation_y_val)
    print('大类训练结束')
    save_model(formation_model, save_dir, 'formation_model')

    # 2. 训练小类模型
    small_class_models = {}
    single_class_map = {} # 记录仅有单一小类的大类，避免预测报错

    for idx, formation in enumerate(formation_encoder.classes_):
        # 切片提取该大类的训练集
        mask_train = (formation_y_train == idx)
        X_sub_train = X_train[mask_train]
        y_sub_train = num_train[mask_train]
        
        # 切片提取该大类的验证集
        mask_val = (formation_y_val == idx)
        X_sub_val = X_val[mask_val]
        y_sub_val = num_val[mask_val]
        
        num_encoder = LabelEncoder()
        y_sub_encoded_train = num_encoder.fit_transform(y_sub_train)
        n_classes = len(np.unique(y_sub_encoded_train))
        
        if n_classes == 1:
            print(f"大类[{formation}] 只有一个小类，无需训练模型")
            single_class_map[idx] = y_sub_train.iloc[0] if isinstance(y_sub_train, pd.Series) else y_sub_train[0]
            continue
            
        print(f"大类[{formation}]，小类数: {n_classes}")
        
        # 如果验证集里没有该大类数据，则传 None
        y_sub_encoded_val = num_encoder.transform(y_sub_val) if len(y_sub_val) > 0 else None
        x_val_pass = X_sub_val if len(y_sub_val) > 0 else None
        
        model = train_and_evaluate(X_sub_train, y_sub_encoded_train, n_classes, x_val=x_val_pass, y_val=y_sub_encoded_val)
        
        eng_name = eng_formation_map.get(formation, str(formation)).replace(' ', '')
        save_model(model, save_dir, f'small_class_model_{eng_name}')
        small_class_models[formation] = (model, num_encoder)
        
    print('全部模型保存完毕')
    return formation_model, small_class_models, single_class_map

# ==== 主流程 ====
gfm_path = r'F:/TensorFlow/xinjiang/traindata20250626_2_class_mapping.csv'
eng_formation_map = get_eng_formation_map(gfm_path, class_mapping)

formation_model, small_class_models, single_class_map = train_and_save_all_models(
    X_train, num_train, formation_y_train, 
    X_val, num_val, formation_y_val, 
    formation_encoder, eng_formation_map, model_dir
)

[全局基准提示] 训练集全局盲猜小类基准: 0.2005
[全局基准提示] 验证集全局盲猜小类基准: 0.2007
大类（Formation）类别数: 9
  [基准提示] 训练集盲猜基准: 0.2626
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where
训练轮数: 282, 最终训练精度: 0.6817 (比基准提升: +0.4191), 损失: 0.0248
  >>> 独立验证集精度: 0.6165 <<<
大类训练结束
模型已保存到: F:\TensorFlow\xinjiang\models20260516\formation_model.h5
大类[丛生草类草原]，小类数: 7
  [基准提示] 训练集盲猜基准: 0.2757
训练轮数: 500, 最终训练精度: 0.9459 (比基准提升: +0.6703), 损失: 0.0285
  >>> 独立验证集精度: 0.4130 <<<
模型已保存到: F:\TensorFlow\xinjiang\models20260516\small_class_model_TussokSteppe.h5
大类[丛生草类草甸]，小类数: 3
  [基准提示] 训练集盲猜基准: 0.4800
训练轮数: 187, 最终训练精度: 0.8800 (比基准提升: +0.4000), 损失: 0.3149
  >>> 独立验证集精度: 0.2308 <<<
模型已保存到: F:\TensorFlow\xinjiang\models20260516\small_class_model_TussokMeadow.h5
大类[农业植被] 只有一个小类，无需训练模型
大类[半乔木与灌木荒漠]，小类数: 3
  [基准提示] 训练集盲猜基准: 0.4839
训练轮数: 176, 最终训练精度: 0.7016 (比基准提升: +0.2177), 损失: 0.3478
 

### 消融实验

In [5]:
# =========================================================
# 消融实验 (Ablation Studies) 独立验证对比区 - 深度分析版 (全分层架构)
# =========================================================
print("\n" + "="*60)
print("开始进行网络结构和策略的深度消融实验 (后台静默训练中，请稍候)...")
print("="*60)

# 编码准备
global_num_encoder = LabelEncoder()
num_train_encoded = global_num_encoder.fit_transform(num_train)
num_val_encoded = global_num_encoder.transform(num_val)
total_small_classes = len(global_num_encoder.classes_)

# ----------------- 预处理：构建类别的辅助映射与统计 -----------------
# 构建 num(小类) -> Formation(大类) 的字典映射
num_to_fmt_dict = dict(zip(class_mapping['num'], class_mapping['Formation']))

# 统计验证集各类别的样本量，划分为“多数类”和“少数类” (以中位数为界)
val_class_counts = pd.Series(num_val_encoded).value_counts()
median_count = val_class_counts.median()
majority_classes = val_class_counts[val_class_counts >= median_count].index
minority_classes = val_class_counts[val_class_counts < median_count].index

def get_subset_acc(preds, true_labels, subset_classes):
    """辅助函数：计算特定类别子集（如只有少数类）的准确率"""
    mask = np.isin(true_labels, subset_classes)
    if np.sum(mask) == 0: return 0.0
    return np.mean(preds[mask] == true_labels[mask])

# ================= 封装分层训练与预测流程 =================
def evaluate_hierarchical_ablation(loss_type='focal', use_bn_dropout=True):
    """根据指定的配置，进行完整的分层训练和预测"""
    # 1. 训练大类
    num_formation_classes = len(np.unique(formation_y_train))
    fmt_model = train_and_evaluate(X_train, formation_y_train, num_formation_classes, 
                                   x_val=X_val, y_val=formation_y_val, 
                                   loss_type=loss_type, use_bn_dropout=use_bn_dropout, quiet=True)
    
    # 2. 训练小类
    sub_models = {}
    s_class_map = {}
    for idx, formation in enumerate(formation_encoder.classes_):
        mask_train = (formation_y_train == idx)
        X_sub_train = X_train[mask_train]
        y_sub_train = num_train[mask_train]
        
        mask_val = (formation_y_val == idx)
        X_sub_val = X_val[mask_val]
        y_sub_val = num_val[mask_val]
        
        n_enc = LabelEncoder()
        y_sub_encoded_train = n_enc.fit_transform(y_sub_train)
        n_cls = len(np.unique(y_sub_encoded_train))
        
        if n_cls == 1:
            s_class_map[idx] = y_sub_train.iloc[0] if isinstance(y_sub_train, pd.Series) else y_sub_train[0]
            continue
            
        y_sub_encoded_val = n_enc.transform(y_sub_val) if len(y_sub_val) > 0 else None
        x_val_pass = X_sub_val if len(y_sub_val) > 0 else None
        
        s_model = train_and_evaluate(X_sub_train, y_sub_encoded_train, n_cls, 
                                     x_val=x_val_pass, y_val=y_sub_encoded_val, 
                                     loss_type=loss_type, use_bn_dropout=use_bn_dropout, quiet=True)
        sub_models[idx] = (s_model, n_enc)
        
    # 3. 组合预测
    fmt_preds = np.argmax(fmt_model.predict(X_val, verbose=0), axis=1)
    h_preds = np.full(len(X_val), -9999)
    
    for fmt_idx in np.unique(fmt_preds):
        mask = (fmt_preds == fmt_idx)
        X_sub = X_val[mask]
        if fmt_idx in sub_models:
            s_model, s_enc = sub_models[fmt_idx]
            s_preds = np.argmax(s_model.predict(X_sub, verbose=0), axis=1)
            h_preds[mask] = s_enc.inverse_transform(s_preds)
        elif fmt_idx in s_class_map:
            h_preds[mask] = s_class_map[fmt_idx]
            
    # 返回编码后的预测结果，以兼容原有的评估逻辑
    return global_num_encoder.transform(h_preds)

# ================= 实验 1：分层 vs 不分层 =================
# 1. 基础分层模型 (Hierarchical - Focal, 有 BN/Dropout)
# 复用上一个 Cell 已经训练好的模型结果，避免重复耗时计算
fmt_preds_base = np.argmax(formation_model.predict(X_val, verbose=0), axis=1)
hierarchical_final_preds = np.full(len(X_val), -9999)

for fmt_idx in np.unique(fmt_preds_base):
    fmt_name = formation_encoder.classes_[fmt_idx]
    mask = (fmt_preds_base == fmt_idx)
    X_sub = X_val[mask]
    if fmt_name in small_class_models:
        sub_model, sub_enc = small_class_models[fmt_name]
        sub_preds = np.argmax(sub_model.predict(X_sub, verbose=0), axis=1)
        hierarchical_final_preds[mask] = sub_enc.inverse_transform(sub_preds)
    elif fmt_idx in single_class_map:
        hierarchical_final_preds[mask] = single_class_map[fmt_idx]

acc_small_hierarchical = np.mean(hierarchical_final_preds == num_val)
acc_fmt_hierarchical = np.mean(fmt_preds_base == formation_y_val)

# 生成基准分层模型的 encoded 预测值，供后续评估使用
hierarchical_preds_encoded_base = global_num_encoder.transform(hierarchical_final_preds)

# 2. 不分层单模型 (Flat)
flat_model_base = train_and_evaluate(X_train, num_train_encoded, total_small_classes, 
                                     x_val=X_val, y_val=num_val_encoded, 
                                     loss_type='focal', use_bn_dropout=True, quiet=True)
acc_small_flat = flat_model_base._val_acc

# 大类准确率 (将Flat预测的小类反向映射回大类)
flat_preds_encoded = np.argmax(flat_model_base.predict(X_val, verbose=0), axis=1)
flat_preds_num = global_num_encoder.inverse_transform(flat_preds_encoded)
flat_preds_fmt = [num_to_fmt_dict.get(n, "Unknown") for n in flat_preds_num]
true_fmt = formation_encoder.inverse_transform(formation_y_val)
acc_fmt_flat = np.mean(np.array(flat_preds_fmt) == np.array(true_fmt))

print("\n[实验 1] 分层结构 vs 不分层结构")
print(f"  -> 小类(最终)准确率   | 分层: {acc_small_hierarchical:.4f}, 不分层: {acc_small_flat:.4f}")
print(f"  -> 大类(Formation)准确率 | 分层: {acc_fmt_hierarchical:.4f}, 不分层: {acc_fmt_flat:.4f}")


# ================= 实验 2：(分层结构下) Focal Loss vs CCE =================
hierarchical_preds_encoded_cce = evaluate_hierarchical_ablation(loss_type='cce', use_bn_dropout=True)

acc_focal_h = acc_small_hierarchical # 基准就是使用了 Focal 的分层模型
acc_cce_h = np.mean(hierarchical_preds_encoded_cce == num_val_encoded)

# 多数类 vs 少数类 精度对比 (全在分层结构下对比)
cce_majority_acc = get_subset_acc(hierarchical_preds_encoded_cce, num_val_encoded, majority_classes)
cce_minority_acc = get_subset_acc(hierarchical_preds_encoded_cce, num_val_encoded, minority_classes)

focal_majority_acc = get_subset_acc(hierarchical_preds_encoded_base, num_val_encoded, majority_classes)
focal_minority_acc = get_subset_acc(hierarchical_preds_encoded_base, num_val_encoded, minority_classes)

# 宏平均召回率
macro_recall_cce = recall_score(num_val_encoded, hierarchical_preds_encoded_cce, average='macro', zero_division=0)
macro_recall_focal = recall_score(num_val_encoded, hierarchical_preds_encoded_base, average='macro', zero_division=0)

print("\n[实验 2] 分层结构下：Focal Loss vs CCE (CrossEntropy) Loss")
print(f"  -> 分层最终准确率  | Focal: {acc_focal_h:.4f}, CCE: {acc_cce_h:.4f}")
print(f"  -> 多数类样本准确率 | Focal: {focal_majority_acc:.4f}, CCE: {cce_majority_acc:.4f}")
print(f"  -> 少数类样本准确率 | Focal: {focal_minority_acc:.4f}, CCE: {cce_minority_acc:.4f}")
print(f"  -> 宏平均类准确率   | Focal: {macro_recall_focal:.4f}, CCE: {macro_recall_cce:.4f}")


# ================= 实验 3：(分层结构下) 正则化组件作用 =================
hierarchical_preds_encoded_noreg = evaluate_hierarchical_ablation(loss_type='focal', use_bn_dropout=False)
acc_no_reg_h = np.mean(hierarchical_preds_encoded_noreg == num_val_encoded)

print("\n[实验 3] 分层结构下：使用 BN & Dropout vs 移除 BN & Dropout")
print(f"  -> 分层最终准确率 | BN/Dropout: {acc_focal_h:.4f}, 无 BN/Dropout: {acc_no_reg_h:.4f}")

print("\n" + "="*60)
print("消融实验对比结束。")
print("="*60)


开始进行网络结构和策略的深度消融实验 (后台静默训练中，请稍候)...

[实验 1] 分层结构 vs 不分层结构
  -> 小类(最终)准确率   | 分层: 0.3907, 不分层: 0.3584
  -> 大类(Formation)准确率 | 分层: 0.6165, 不分层: 0.5591

[实验 2] 分层结构下：Focal Loss vs CCE (CrossEntropy) Loss
  -> 分层最终准确率  | Focal: 0.3907, CCE: 0.4086
  -> 多数类样本准确率 | Focal: 0.4118, CCE: 0.4538
  -> 少数类样本准确率 | Focal: 0.2683, CCE: 0.1463
  -> 宏平均类准确率   | Focal: 0.2840, CCE: 0.2396

[实验 3] 分层结构下：使用 BN & Dropout vs 移除 BN & Dropout
  -> 分层最终准确率 | BN/Dropout: 0.3907, 无 BN/Dropout: 0.1613

消融实验对比结束。


### 预测